# JetBot Line Follower: Midterm Progress Notebook

**ENGR-UH 3530 Embedded Systems**
Loubna Kahramane (lk3182), Changju Song (cs6514), Isaac Kim (ik2267), Euisun Choi (ec4495)

This notebook drives a Waveshare JetBot (Jetson Nano 4 GB, CSI camera) along a black line, using the camera as the only sensor. It handles curves, sharp bends, gaps in the tape and green intersection markers (left, right and dead end).

### How the code is organised

Every camera frame goes through the same path:

1. **Perception** (Section 3) turns the image into an `Observation`: the line position in three horizontal bands, the row of any junction crossbar, and a green-marker decision.
2. **NavigationCoordinator** (Section 7) hands the frame to exactly one controller, in priority order:
   - `GreenMarkerTurnController`: confirms a marker, stops, advances over the junction and turns.
   - `GapCrossingController`: crosses a gap in the tape after a straight approach.
   - `BlackLineController` (inside the gap controller): normal line following, pivots on sharp bends and a short search if the line is lost.
3. **The runner** (Section 8) is the only part of the program that sends commands to the motors. It also stops the robot if the camera stops delivering fresh frames.

All robot-specific values are measured and kept in Section 2. Most time limits are calculated from those measurements rather than tuned one by one.

### Running the notebook

Run the cells in order from the top. Nothing drives until `run_robot()` is called in Section 9. Use `run_preview()` first to check perception with the motors off, and run the cleanup cell before switching to another notebook.

## 1. Hardware and camera

Creates the robot and camera objects and shows a live camera view. The motors are stopped immediately as a safety measure.

In [ ]:
import time, csv
from collections import deque, namedtuple
import cv2
import numpy as np
import traitlets
import ipywidgets as widgets
from IPython.display import display
from jetbot import Camera, Robot, bgr8_to_jpeg

robot = Robot()
robot.stop()

# 224 x 224 is enough to see the tape clearly and keeps image processing fast
camera = Camera.instance(width=224, height=224)
live_view = widgets.Image(format="jpeg", width=336, height=336)
camera_link = traitlets.dlink((camera, "value"), (live_view, "value"), transform=bgr8_to_jpeg)
display(live_view)

The camera callback keeps only the newest frame together with the time it arrived. Frames are never queued, so the control loop always works on the most recent image, and the timestamp lets it detect a stalled camera.

In [ ]:
latest = {"sample": None}   # (frame, arrival time) of the newest image only

def receive_frame(change):
    latest["sample"] = (change["new"], time.monotonic())

camera.observe(receive_frame, names="value")
time.sleep(0.5)
print("Camera frame received." if latest["sample"] is not None else "No camera frame yet.")

Camera frame received.


## 2. Calibration values

Everything that depends on the physical robot is kept here. Motor values are normalised commands between -1 and 1.

The three measured quantities are:

- `ROTATION_TIME_360`: seconds for one full turn in place at `GREEN_TURN_SPEED` (measured with `spin_test()` below).
- `CAMERA_BLIND_TIME`: time for tape that has just left the bottom of the image to reach the wheel axle.
- `MIN_DRIVE_SPEED`: the lowest command that still moves the robot reliably on the floor.

Re-run this cell, and every cell below it, after changing any value.

In [ ]:
# Motor commands
BASE_SPEED = 0.12          # forward speed on straight tape
CURVE_SPEED = 0.12         # forward speed in bends
MIN_DRIVE_SPEED = 0.12     # below this the motors stall
KP = 0.14                  # proportional steering gain
MAX_SPEED = 0.25           # upper limit for any wheel command
TURN_SPEED = 0.15          # outer wheel during a black-line pivot
GREEN_TURN_SPEED = 0.15    # both wheels (one reversed) during a green-marker turn
PIVOT_INNER_RATIO = -0.6   # inner wheel runs backwards at 60% so the robot turns near its axle

# Measured timing
ROTATION_TIME_360 = 2.15   # s per full turn in place at GREEN_TURN_SPEED
CAMERA_BLIND_TIME = 0.12   # s from tape leaving the image to tape under the axle
FRAME_TIMEOUT = 0.5        # s; an older camera frame stops the robot

# Derived values
QUARTER_TURN_TIME = ROTATION_TIME_360 / 4.0
# A black-line pivot turns more slowly than a green turn, because the inner wheel only runs
# at 60% in reverse. Scale the rotation time by the ratio of wheel-speed differences.
LINE_PIVOT_TIME_360 = ROTATION_TIME_360 * (2.0 * GREEN_TURN_SPEED) / ((1.0 - PIVOT_INNER_RATIO) * TURN_SPEED)

def clamp(value, low, high):
    return max(low, min(high, value))

**Rotation test.** Spins the robot in place for a fixed time. Measure the angle turned, then set `ROTATION_TIME_360 = seconds * 360 / angle`. The call is commented out so that running the whole notebook does not move the robot.

In [ ]:
def spin_test(seconds=2.15):
    try:
        robot.set_motors(-GREEN_TURN_SPEED, GREEN_TURN_SPEED)
        time.sleep(seconds)
    finally:
        robot.stop()

# spin_test(2.15)

## 3. Perception

`perceive()` turns one camera frame into an `Observation`:

- **Line position**: the image is split into three horizontal bands (near, middle, far). Each band reports where the tape is, from -1 (left edge) to +1 (right edge), or `None` if no tape is visible.
- **Junction row**: the lowest row that contains a wide horizontal stroke of tape, such as an intersection crossbar.
- **Marker decision**: `LEFT`, `RIGHT`, `BOTH`, `CENTER`, `UNKNOWN` or `NONE`. `CENTER` means markers are visible but none of them is a valid instruction right now.

Two details matter at junctions. A crossbar inside a band would pull the average tape position sideways, so in that case the band reports the position of the line the robot is driving along instead. And a green marker only counts when a crossbar is visible beyond it, because the rules place markers just before the intersection. This ignores markers that have already been passed or that belong to the other direction.

In [ ]:
BLACK_THRESHOLD = 70   # grey level; darker pixels count as tape
# (name, top, bottom) as fractions of the image height
BANDS = [("near", 0.72, 0.88), ("middle", 0.54, 0.70), ("far", 0.36, 0.52)]

# Junction detection
CROSS_BAND_WIDTH = 0.40     # a tape blob wider than 40% of the image contains a crossbar
STROKE_FILL = 0.85          # a column that is 85% black over the band height belongs to the through-line
CROSS_ROW_FRACTION = 0.25   # a row that is more than 25% black is part of a horizontal stroke
CROSS_MIN_ROWS = 3          # the stroke must be at least this many rows thick
CROSS_SEARCH = (0.30, 0.95) # vertical range searched for a crossbar

# Green markers (OpenCV HSV: hue runs from 0 to 179)
GREEN_LOWER = np.array([35, 70, 50], dtype=np.uint8)
GREEN_UPPER = np.array([90, 255, 255], dtype=np.uint8)
GREEN_MIN_AREA = 40
GREEN_MAX_AREA = 3000
GREEN_ROI_TOP = 0.45          # markers are only searched for in this part of the image
GREEN_ROI_BOTTOM = 0.92
GREEN_TRIGGER_Y = 0.65        # a marker must be this far down the image to be acted on
GREEN_SIDE_MARGIN = 8.0       # px; closer than this to the line, the side is ambiguous
GREEN_MAX_SIDE_DISTANCE = 65.0  # px; further than this, the marker belongs to another branch
GREEN_PAIR_Y_TOLERANCE = 0.15   # two markers must be roughly level to count as a dead end

Observation = namedtuple("Observation", "positions direction junction_y zone_markers markers crossing")


def detect_green(hsv, height):
    # Threshold on hue, then remove specks and fill small holes
    colour = cv2.inRange(hsv, GREEN_LOWER, GREEN_UPPER)
    kernel = np.ones((3, 3), dtype=np.uint8)
    colour = cv2.morphologyEx(colour, cv2.MORPH_OPEN, kernel)
    colour = cv2.morphologyEx(colour, cv2.MORPH_CLOSE, kernel)

    roi = colour.copy()
    roi[:int(height * GREEN_ROI_TOP), :] = 0
    roi[int(height * GREEN_ROI_BOTTOM):, :] = 0

    contours = cv2.findContours(roi, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
    markers = []
    for contour in contours:
        area = cv2.contourArea(contour)
        x, y, w, h = cv2.boundingRect(contour)
        # Keep roughly square blobs of a plausible size
        if not GREEN_MIN_AREA <= area <= GREEN_MAX_AREA or h == 0 or not 0.4 <= w / float(h) <= 2.5:
            continue
        m = cv2.moments(contour)
        if m["m00"] <= 0:
            continue
        markers.append(dict(x=m["m10"] / m["m00"], y=m["m01"] / m["m00"], area=area, box=(x, y, w, h)))
    markers.sort(key=lambda item: item["area"], reverse=True)
    return markers, colour, roi


def through_line_x(strip):
    # Column of the tape that runs through the whole band, used when a crossbar is in the band.
    # Returns None if there is no such line (for example at a corner).
    black = strip > 0
    height, width = black.shape

    # Case 1: a vertical line. Find columns that are black over most of the band height.
    fill = black.mean(axis=0)
    cols = np.flatnonzero(fill >= STROKE_FILL)

    if cols.size:
        runs = np.split(cols, np.flatnonzero(np.diff(cols) > 1) + 1)
        return float(max(runs, key=len).mean())

    # Case 2: a slightly tilted line. Pair tape segments on the top and bottom edges of the band
    # and check whether a straight path between them is black on almost every row.
    if height < 8:
        return None

    edge_rows = max(2, height // 8)
    max_line_width = 0.25 * width

    def edge_runs(edge):
        cols = np.flatnonzero(edge.mean(axis=0) >= 0.75)
        if cols.size == 0:
            return []
        runs = np.split(cols, np.flatnonzero(np.diff(cols) > 1) + 1)
        # A wide run is part of the crossbar, not the end of the through-line
        return [r for r in runs if 2 <= len(r) <= max_line_width]

    top_runs = edge_runs(black[:edge_rows])
    bottom_runs = edge_runs(black[-edge_rows:])

    if not top_runs or not bottom_runs:
        return None

    y_top = (edge_rows - 1) / 2.0
    y_bottom = height - 1 - y_top
    rows = np.arange(height)
    candidates = []

    for bottom in bottom_runs:
        for top in top_runs:
            x_bottom = float(bottom.mean())
            x_top = float(top.mean())

            slope = (x_bottom - x_top) / (y_bottom - y_top)

            # Only small tilts are handled here; sharp bends are left to the line controller
            if abs(slope) > 0.8:
                continue

            xs = x_top + slope * (rows - y_top)
            supported = np.zeros(height, dtype=bool)

            # Allow +/- 2 px for noise and the width of the tape
            for offset in (-2, -1, 0, 1, 2):
                xi = np.rint(xs + offset).astype(int)
                valid = (xi >= 0) & (xi < width)
                supported[valid] |= black[rows[valid], xi[valid]]

            coverage = float(supported.mean())
            if coverage < STROKE_FILL:
                continue

            midpoint = x_top + slope * ((height - 1) / 2.0 - y_top)
            candidates.append((coverage, midpoint))

    if not candidates:
        return None

    # If the candidates disagree, the result is unreliable, so fall back to the band average
    centers = [candidate[1] for candidate in candidates]
    if max(centers) - min(centers) > 0.08 * width:
        return None

    return float(max(candidates, key=lambda item: item[0])[1])


def find_junction_y(mask, height, width):
    # Lowest horizontal stroke (crossbar or corner) as a fraction of image height, or None
    r0, r1 = int(CROSS_SEARCH[0] * height), int(CROSS_SEARCH[1] * height)
    counts = (mask[r0:r1] > 0).sum(axis=1)
    rows = np.flatnonzero(counts > CROSS_ROW_FRACTION * width)
    if rows.size == 0:
        return None
    runs = [r for r in np.split(rows, np.flatnonzero(np.diff(rows) > 1) + 1) if len(r) >= CROSS_MIN_ROWS]
    if not runs:
        return None
    return (r0 + float(runs[-1].mean())) / height


def detect_line(frame, gray, green_colour):
    height, width = frame.shape[:2]
    _, mask = cv2.threshold(gray, BLACK_THRESHOLD, 255, cv2.THRESH_BINARY_INV)
    # Dark green tape can fall under the black threshold, so remove it from the black mask
    mask[cv2.dilate(green_colour, np.ones((5, 5), np.uint8)) > 0] = 0

    overlay = frame.copy()
    positions, crossing = {}, set()
    for name, top, bottom in BANDS:
        y0, y1 = int(top * height), int(bottom * height)
        strip = mask[y0:y1, :]
        contours = cv2.findContours(strip, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
        positions[name] = None
        if contours:
            contour = max(contours, key=cv2.contourArea)
            fraction = cv2.contourArea(contour) / float(strip.size)
            m = cv2.moments(contour)
            # Ignore tiny specks and a band that is almost entirely dark
            if 0.005 < fraction < 0.80 and m["m00"] > 0:
                cx = m["m10"] / m["m00"]
                # A very wide blob contains a crossbar: use the through-line instead of the average
                if cv2.boundingRect(contour)[2] > CROSS_BAND_WIDTH * width:
                    sx = through_line_x(strip)
                    if sx is not None:
                        cx = sx
                        crossing.add(name)
                positions[name] = (cx - width / 2.0) / (width / 2.0)   # -1 = left edge, +1 = right edge
                cv2.circle(overlay, (int(cx), (y0 + y1) // 2), 4, (0, 255, 0), -1)
        cv2.rectangle(overlay, (0, y0), (width - 1, y1), (255, 0, 0), 1)
    return positions, crossing, mask, overlay


def get_line_points(positions, width, height):
    # Pixel coordinates of the line in each visible band, ordered far -> near
    band_y = {name: (top + bottom) / 2.0 for name, top, bottom in BANDS}
    return [(width * (1.0 + positions[name]) / 2.0, height * band_y[name])
            for name in ("far", "middle", "near") if positions.get(name) is not None]


def classify_green_direction(positions, markers, width, height, junction_y):
    # Returns the marker decision and the number of markers in the trigger zone
    zone = [m for m in markers if m["y"] >= height * GREEN_TRIGGER_Y]
    if not markers:
        return "NONE", 0

    # The side of a marker is judged against the direction of the line, so at least two
    # bands must see the tape
    points = get_line_points(positions, width, height)
    if len(points) < 2:
        return "UNKNOWN", len(zone)
    near_x, near_y = points[-1]
    dx, dy = points[0][0] - near_x, points[0][1] - near_y
    length = (dx * dx + dy * dy) ** 0.5
    if length < 1e-6:
        return "UNKNOWN", len(zone)
    dx, dy = dx / length, dy / length

    sides = {"LEFT": [], "RIGHT": []}
    for marker in zone:
        # Rule 1.5.6: a valid marker sits just before the junction, so a crossbar must be visible
        # beyond it. This ignores markers that have already been passed.
        if junction_y is None or marker["y"] <= junction_y * height:
            continue
        # Cross product: positive = left of the line, negative = right
        score = (marker["x"] - near_x) * dy - (marker["y"] - near_y) * dx
        if GREEN_SIDE_MARGIN < abs(score) <= GREEN_MAX_SIDE_DISTANCE:
            sides["LEFT" if score > 0 else "RIGHT"].append(marker)

    if sides["LEFT"] and sides["RIGHT"]:
        # Two markers count as a dead end only if they are roughly level with each other
        paired = any(abs(a["y"] - b["y"]) <= height * GREEN_PAIR_Y_TOLERANCE
                     for a in sides["LEFT"] for b in sides["RIGHT"])
        return ("BOTH" if paired else "UNKNOWN"), len(zone)
    if sides["LEFT"]:
        return "LEFT", len(zone)
    if sides["RIGHT"]:
        return "RIGHT", len(zone)
    return "CENTER", len(zone)


def perceive(frame):
    height, width = frame.shape[:2]
    gray = cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    markers, green_colour, green_roi = detect_green(hsv, height)
    positions, crossing, mask, overlay = detect_line(frame, gray, green_colour)
    junction_y = find_junction_y(mask, height, width)
    direction, zone = classify_green_direction(positions, markers, width, height, junction_y)
    return Observation(positions, direction, junction_y, zone, markers, crossing), overlay, green_roi

### 3.1 Perception self-test

Draws nine simple test images (straight line, junctions, corners and marker layouts) and checks that perception gives the expected result. No camera or motors are needed. Run this after changing any threshold above.

In [ ]:
def synth(lines=(), markers=(), thickness=10):
    # White 224 x 224 image with black lines and green squares
    img = np.full((224, 224, 3), 235, np.uint8)
    for x0, y0, x1, y1 in lines:
        cv2.line(img, (x0, y0), (x1, y1), (20, 20, 20), thickness)
    for x, y in markers:
        cv2.rectangle(img, (x - 7, y - 7), (x + 7, y + 7), (40, 170, 40), -1)
    return img


def self_test(verbose=True):
    V = (112, 0, 112, 223)   # vertical line through the image centre
    cases = [
        ("straight line", synth([V]), lambda o: abs(o.positions["middle"]) < 0.05 and o.junction_y is None),
        ("4-way: middle band not pulled sideways", synth([V, (0, 140, 223, 140)]),
         lambda o: abs(o.positions["middle"]) < 0.05 and o.junction_y is not None),
        ("T with left branch: keep through-line", synth([V, (0, 140, 112, 140)]),
         lambda o: abs(o.positions["middle"]) < 0.05),
        ("unmarked left corner: still steer left", synth([(112, 223, 112, 140), (112, 140, 0, 140)]),
         lambda o: o.positions["middle"] is not None and o.positions["middle"] < -0.2),
        ("left marker before junction", synth([V, (0, 140, 223, 140)], [(94, 156)]),
         lambda o: o.direction == "LEFT"),
        ("right marker before junction", synth([V, (0, 140, 223, 140)], [(130, 156)]),
         lambda o: o.direction == "RIGHT"),
        ("dead end (both markers)", synth([V, (0, 140, 223, 140)], [(94, 156), (130, 156)]),
         lambda o: o.direction == "BOTH"),
        ("marker beyond the crossbar is ignored", synth([V, (0, 175, 223, 175)], [(94, 157)]),
         lambda o: o.direction not in ("LEFT", "RIGHT", "BOTH")),
        ("marker with no junction is ignored", synth([V], [(94, 170)]),
         lambda o: o.direction not in ("LEFT", "RIGHT", "BOTH")),
    ]
    passed = 0
    for name, img, check in cases:
        obs = perceive(img)[0]
        ok = bool(check(obs))
        passed += ok
        if verbose:
            pos = {k: (None if v is None else round(v, 2)) for k, v in obs.positions.items()}
            jy = None if obs.junction_y is None else round(obs.junction_y, 2)
            print("%-42s %s  dir=%-7s junction=%s %s" % (name, "PASS" if ok else "FAIL", obs.direction, jy, pos))
    print("%d/%d passed" % (passed, len(cases)))
    return passed == len(cases)

self_test()

straight line                              PASS  dir=NONE    junction=None {'near': 0.0, 'middle': 0.0, 'far': 0.0}
4-way: middle band not pulled sideways     PASS  dir=NONE    junction=0.62 {'near': 0.0, 'middle': 0.0, 'far': 0.0}
T with left branch: keep through-line      PASS  dir=NONE    junction=0.62 {'near': 0.0, 'middle': 0.0, 'far': 0.0}
unmarked left corner: still steer left     PASS  dir=NONE    junction=0.62 {'near': 0.0, 'middle': -0.44, 'far': None}
left marker before junction                PASS  dir=LEFT    junction=0.62 {'near': 0.0, 'middle': 0.0, 'far': 0.0}
right marker before junction               PASS  dir=RIGHT   junction=0.62 {'near': 0.0, 'middle': 0.0, 'far': 0.0}
dead end (both markers)                    PASS  dir=BOTH    junction=0.62 {'near': 0.0, 'middle': 0.0, 'far': 0.0}
marker beyond the crossbar is ignored      PASS  dir=CENTER  junction=0.78 {'near': 0.0, 'middle': 0.0, 'far': 0.0}
marker with no junction is ignored         PASS  dir=CENTER  junction

## 4. BlackLineController

Normal line following. The steering error is a weighted average of the three band positions (near 0.45, middle 0.35, far 0.20), plus a look-ahead term that starts the turn before a bend reaches the robot. The wheel speeds are set by proportional control.

| State | What the robot does | When it leaves the state |
|---|---|---|
| `FOLLOW` | Proportional steering | Error of 0.55 or more starts a pivot; losing all bands starts a search |
| `FOLLOW_SLOW` | Near and far bands point in opposite directions (a zig-zag), so the far band is ignored and speed drops to the minimum | The bands agree again |
| `TURN` | Pivots near the axle, inner wheel reversed | Near band back within 0.22 of centre |
| `SEARCH` | Line lost: pivots towards the side the line was last seen | Tape seen again, or about 120° without success |
| `STOP` | Motors off for the rest of the run | — |

A pivot is stopped if the error stops shrinking for 0.8 s or the robot has turned about 180°.

In [ ]:
TURN_ENTER = 0.55               # start a pivot at this steering error
TURN_EXIT = 0.22                # end it when the near band is this close to centre
TURN_EXIT_MIDDLE = 0.35         # middle band only needs to be roughly centred (the next bend may be in view)
LOOKAHEAD_GAIN = 0.40
ZIGZAG_SPREAD = 0.60            # near and far this far apart on opposite sides = two bends in view
TURN_PROGRESS = 0.03            # error must shrink by at least this much to count as progress
TURN_STALL_TIMEOUT = 0.8        # s without progress before giving up
TURN_HARD_TIMEOUT = 0.5 * LINE_PIVOT_TIME_360   # never pivot more than about 180 degrees
SEARCH_TIMEOUT = LINE_PIVOT_TIME_360 / 3.0      # search for at most about 120 degrees


class BlackLineController:
    # Follows black tape, pivots on sharp bends and searches briefly when the tape is lost

    def __init__(self):
        self.turn_direction = 0     # +1 right, -1 left
        self.turn_started = None    # time the current pivot started, or None
        self.lost_started = None    # time the line was lost, or None
        self.last_direction = 0     # side the line was last seen on
        self.stopped = False
        self.best_turn_error = None
        self.last_progress = None

    def stop(self, reason):
        self.stopped = True
        return 0.0, 0.0, "STOP: " + reason

    def pivot(self, direction):
        # Outer wheel forward, inner wheel backwards so the robot turns close to its axle
        speed = clamp(TURN_SPEED, 0.0, MAX_SPEED)
        inner = clamp(PIVOT_INNER_RATIO * speed, -MAX_SPEED, 0.0)
        return (speed, inner) if direction > 0 else (inner, speed)

    def start_turn(self, direction, now, error=1.0):
        self.turn_started = now
        self.turn_direction = direction
        self.best_turn_error = abs(error)
        self.last_progress = now

    def step(self, positions, now):
        if self.stopped:
            return 0.0, 0.0, "STOP: controller stopped"
        near, middle = positions.get("near"), positions.get("middle")
        visible = [(name, value) for name, value in positions.items() if value is not None]

        # No tape in any band: search towards the side it was last seen on, for a limited time
        if not visible:
            if self.lost_started is None:
                self.lost_started = now
            if not self.last_direction:
                return self.stop("line lost without a known turn direction")
            if now - self.lost_started >= SEARCH_TIMEOUT:
                return self.stop("line not reacquired")
            if self.turn_started is None:
                self.start_turn(self.last_direction, now)
            if now - self.turn_started >= TURN_HARD_TIMEOUT:
                return self.stop("turn timed out")
            left, right = self.pivot(self.turn_direction)
            return left, right, "SEARCH"

        returning = self.lost_started is not None
        self.lost_started = None

        # Steering error from the visible bands
        weights = {"near": 0.45, "middle": 0.35, "far": 0.20}
        farthest = positions.get("far")
        if farthest is None:
            farthest = middle
        zigzag = (near is not None and farthest is not None and near * farthest < 0
                  and abs(farthest - near) >= ZIGZAG_SPREAD)
        if zigzag:
            # The far band is showing the second bend; steer on what is directly ahead
            used = [(n, v) for n, v in visible if n != "far"]
            error = sum(weights[n] * v for n, v in used) / sum(weights[n] for n, _ in used)
        else:
            error = sum(weights[n] * v for n, v in visible) / sum(weights[n] for n, _ in visible)
            if near is not None and farthest is not None:
                error += LOOKAHEAD_GAIN * (farthest - near)   # start turning before the bend arrives
        error = clamp(error, -1.0, 1.0)
        if abs(error) > 0.15:
            self.last_direction = 1 if error > 0 else -1

        # A pivot in progress: finish it once the line is centred, give up if it stops improving
        if self.turn_started is not None:
            if returning:
                # The line has just reappeared after a search, so restart progress tracking
                self.best_turn_error, self.last_progress = abs(error), now
            if near is not None and abs(near) < TURN_EXIT and (middle is None or abs(middle) < TURN_EXIT_MIDDLE):
                self.turn_started = None
                self.turn_direction = 0
            else:
                if abs(error) < self.best_turn_error - TURN_PROGRESS:
                    self.best_turn_error, self.last_progress = abs(error), now
                if now - self.last_progress >= TURN_STALL_TIMEOUT:
                    return self.stop("turn made no progress")
                if now - self.turn_started >= TURN_HARD_TIMEOUT:
                    return self.stop("turn timed out")

        # Separate enter and exit thresholds stop the robot switching back and forth
        if self.turn_started is None and abs(error) >= TURN_ENTER:
            self.start_turn(1 if error > 0 else -1, now, error)

        if self.turn_started is not None:
            left, right = self.pivot(self.turn_direction)
            return left, right, "TURN"

        # Normal proportional steering; slow down in bends but never below the stall speed
        bend = min(abs(error) / TURN_ENTER, 1.0)
        base = BASE_SPEED + (CURVE_SPEED - BASE_SPEED) * bend
        base = MIN_DRIVE_SPEED if zigzag else max(base, MIN_DRIVE_SPEED)
        correction = KP * error
        left = clamp(base + correction, 0.0, MAX_SPEED)
        right = clamp(base - correction, 0.0, MAX_SPEED)
        return left, right, ("FOLLOW_SLOW" if zigzag else "FOLLOW")

## 5. GapCrossingController

A missing line can mean either a gap in the tape or that the robot has drifted off it. This controller tells the two apart using what happened just before the line disappeared.

It contains its own `BlackLineController` and passes every ordinary frame to it. While doing so, it counts recent frames in which the robot was following a straight, centred line. If the tape then vanishes after at least 3 such frames in the last 0.6 s, it drives straight across the gap with a small remembered steering correction. Otherwise the line controller searches as usual.

The crossing ends when the near and middle bands both see tape for 2 frames in a row. If that does not happen within 1.9 s, the robot stops.

In [ ]:
GAP_SPEED = BASE_SPEED
GAP_TIMEOUT = 1.9            # s allowed to find tape on the other side
GAP_STEER_LIMIT = 0.015      # largest steering correction carried across the gap
GAP_APPROACH_FRAMES = 3      # straight frames needed before a gap is believed
GAP_APPROACH_MAX_AGE = 0.6   # s; how recent that straight approach must be
GAP_ALIGNMENT = 0.25         # all band positions within this of centre
GAP_STRAIGHT_SPREAD = 0.12   # bands may differ by at most this much
GAP_CONFIRM_FRAMES = 2       # frames with tape needed to end the crossing


class GapCrossingController:
    # Decides whether a lost line is a gap; otherwise passes frames to the line controller

    def __init__(self):
        self.black_line = BlackLineController()
        self.straight_frames = 0      # recent frames of straight, centred following
        self.straight_seen = None     # time of the last such frame
        self.saved_correction = 0.0   # smoothed steering correction to carry across a gap
        self.gap_started = None       # time the current crossing started, or None
        self.confirm_frames = 0

    def gap_commands(self):
        return (clamp(GAP_SPEED + self.saved_correction, 0.0, MAX_SPEED),
                clamp(GAP_SPEED - self.saved_correction, 0.0, MAX_SPEED),
                "GAP: forward, awaiting tape")

    def step(self, positions, now):
        if self.black_line.stopped:
            return 0.0, 0.0, "STOP: controller stopped"
        near, middle = positions.get("near"), positions.get("middle")
        values = [v for v in positions.values() if v is not None]

        # A crossing in progress: keep driving until tape is seen in two consecutive frames
        if self.gap_started is not None:
            if now - self.gap_started >= GAP_TIMEOUT:
                return self.black_line.stop("gap crossing timed out")
            self.confirm_frames = self.confirm_frames + 1 if (near is not None and middle is not None) else 0
            if self.confirm_frames < GAP_CONFIRM_FRAMES:
                return self.gap_commands()
            # Tape found: reset the line controller's search state and carry on following
            self.gap_started = None
            self.confirm_frames = 0
            self.black_line.lost_started = None
            self.black_line.last_direction = 0

        # No tape visible: start a crossing only after a recent straight approach
        if not values:
            recent = self.straight_seen is not None and now - self.straight_seen <= GAP_APPROACH_MAX_AGE
            if self.black_line.turn_started is None and recent and self.straight_frames >= GAP_APPROACH_FRAMES:
                self.gap_started = now
                self.straight_frames = 0
                self.straight_seen = None
                self.confirm_frames = 0
                return self.gap_commands()
            return self.black_line.step(positions, now)

        # Tape visible: let the line controller drive, and record whether it is a straight approach
        left, right, state = self.black_line.step(positions, now)
        aligned = max(abs(v) for v in values) <= GAP_ALIGNMENT
        straight = len(values) >= 2 and max(values) - min(values) <= GAP_STRAIGHT_SPREAD
        if state == "FOLLOW" and near is not None and aligned and straight:
            if self.straight_seen is None or now - self.straight_seen > GAP_APPROACH_MAX_AGE:
                self.straight_frames = 0
            self.straight_frames += 1
            self.straight_seen = now
            correction = clamp((left - right) / 2.0, -GAP_STEER_LIMIT, GAP_STEER_LIMIT)
            self.saved_correction = 0.5 * self.saved_correction + 0.5 * correction
        elif state != "FOLLOW" or not aligned or (len(values) >= 2 and not straight):
            self.straight_frames = 0
            self.straight_seen = None
        return left, right, state

## 6. GreenMarkerTurnController

Handles green intersection markers. The sequence is:

1. **Confirm**: the same decision (`LEFT`, `RIGHT` or `BOTH`) must appear in 3 of the last 5 frames, so a single bad frame cannot trigger a turn.
2. **Stop** for 0.2 s. If a second marker comes into view while stopped, a left or right turn is changed to a dead-end U-turn.
3. **Advance** until the crossbar leaves the bottom of the image, then drive another `CAMERA_BLIND_TIME` so the axle is over the junction. A dead end skips this step and turns around before the junction.
4. **Turn** in place: fast for the first 75% of the expected angle, then slowly until the line is centred for 3 frames. The expected angle comes from `ROTATION_TIME_360`.
5. **Done**: control returns to line following. A new marker is accepted after a 0.4 s cooldown.

Each step has a time limit, and running past it stops the robot. `step()` returns `None` when no green maneuver is active, which tells the coordinator to use another controller for that frame.

In [ ]:
# Confirmation
GREEN_VOTE_WINDOW = 5          # frames remembered
GREEN_VOTES_NEEDED = 3         # agreeing frames needed to act
GREEN_BOTH_UPGRADE = 2         # BOTH frames seen while stopped that change a turn into a U-turn
GREEN_STOP_TIME = 0.20

# Advance over the junction
GREEN_ADVANCE_SPEED = BASE_SPEED
GREEN_ADVANCE_STEER = 0.05     # small steering correction while advancing
JUNCTION_NEAR_Y = 0.75         # the crossbar must get this low before its disappearance counts
JUNCTION_EXIT_Y = 0.90         # or reach this row
JUNCTION_JUMP = 0.20           # a stroke this much higher is a different, farther junction
GREEN_NO_JUNCTION_WAIT = 0.30  # s; if no crossbar is ever seen, assume it is already under the camera
GREEN_ADVANCE_TIMEOUT = 2.0

# Turn limits, derived from the measured rotation time
GREEN_TURN_MIN_TIME = 0.6 * QUARTER_TURN_TIME
GREEN_TURN_TIMEOUT = 2.5 * QUARTER_TURN_TIME
UTURN_MIN_TIME = 0.75 * 2 * QUARTER_TURN_TIME
UTURN_TIMEOUT = 2.0 * 2 * QUARTER_TURN_TIME
GREEN_REACQUIRE_FRAMES = 3     # centred frames needed to finish the turn

# Re-arming after a turn
GREEN_COOLDOWN = 0.4
GREEN_CLEAR_FRAMES = 0         # cooldown only; old markers are already rejected by perception

# Fast-then-slow turn
GREEN_TURN_SLOW_SPEED = 0.10   # lowest speed that still rotates the robot reliably
GREEN_SLOW_ZONE = 0.6          # slow down once the line is this close to centre
FEEDFORWARD_FRACTION = 0.75    # fraction of the expected turn done at full speed


def line_centered(positions):
    return all(positions.get(n) is not None and abs(positions[n]) < TURN_EXIT for n in ("near", "middle"))


class GreenMarkerTurnController:
    # Confirms green-marker instructions and carries them out.
    # step() returns None when this controller is not in control of the robot.

    def __init__(self):
        self.state = "FOLLOW"          # FOLLOW, GREEN_STOP, GREEN_ADVANCE or GREEN_TURN
        self.state_started = None
        self.direction = None          # LEFT, RIGHT or BOTH
        self.votes = deque(maxlen=GREEN_VOTE_WINDOW)
        self.both_votes = 0
        self.armed = True              # False until the cooldown after a turn has passed
        self.clear_frames = 0
        self.cooldown_until = 0.0
        self.departed = False          # True once the line has left the centre during the turn
        self.reacquire_frames = 0
        self.stopped = False
        self.reason = ""

    def stop(self, reason):
        self.stopped = True
        self.reason = reason
        return 0.0, 0.0, "STOP: " + reason

    def enter(self, state, now):
        self.state = state
        self.state_started = now

    def turn(self, positions, elapsed):
        # Full speed for most of the expected turn, then slow once the line is close to centre
        expected = (2 if self.direction == "BOTH" else 1) * QUARTER_TURN_TIME
        near, middle = positions.get("near"), positions.get("middle")
        line_close = any(v is not None and abs(v) < GREEN_SLOW_ZONE for v in (near, middle))
        slow = elapsed >= FEEDFORWARD_FRACTION * expected and line_close
        speed = clamp(GREEN_TURN_SLOW_SPEED if slow else GREEN_TURN_SPEED, 0.0, MAX_SPEED)
        suffix = "_SLOW" if slow else ""
        if self.direction == "RIGHT":
            return speed, -speed, "GREEN_TURN_RIGHT" + suffix
        label = "GREEN_U_TURN" if self.direction == "BOTH" else "GREEN_TURN_LEFT"
        return -speed, speed, label + suffix

    def confirmed(self):
        # The decision that appears in enough of the recent frames, or None
        counts = {d: self.votes.count(d) for d in ("LEFT", "RIGHT", "BOTH")}
        best = max(counts, key=counts.get)
        return best if counts[best] >= GREEN_VOTES_NEEDED else None

    def begin_advance(self, now):
        self.junction_seen = False
        self.last_junction_y = None
        self.passed_frames = 0
        self.post_started = None
        self.enter("GREEN_ADVANCE", now)

    def begin_turn(self, now):
        self.departed = False
        self.reacquire_frames = 0
        self.enter("GREEN_TURN", now)

    def advance(self, obs, now):
        # Drive forward until the crossbar has passed under the camera, then for CAMERA_BLIND_TIME.
        # Returns a motor command, or None once the axle is over the junction.
        elapsed = now - self.state_started
        if elapsed >= GREEN_ADVANCE_TIMEOUT:
            return self.stop("junction not passed during advance")
        if self.post_started is None:
            jy = obs.junction_y
            # Track the same crossbar as it moves down the image (ignore a farther one appearing)
            if jy is not None and (self.last_junction_y is None or jy >= self.last_junction_y - JUNCTION_JUMP):
                self.junction_seen = True
                self.last_junction_y = jy
                under = jy >= JUNCTION_EXIT_Y
            elif self.junction_seen:
                under = self.last_junction_y >= JUNCTION_NEAR_Y
            else:
                under = elapsed >= GREEN_NO_JUNCTION_WAIT
            self.passed_frames = self.passed_frames + 1 if under else 0
            if self.passed_frames >= 2:
                self.post_started = now
        elif now - self.post_started >= CAMERA_BLIND_TIME:
            return None
        near = obs.positions.get("near")
        steer = clamp(KP * near, -GREEN_ADVANCE_STEER, GREEN_ADVANCE_STEER) if near is not None else 0.0
        speed = clamp(GREEN_ADVANCE_SPEED, 0.0, MAX_SPEED)
        label = "GREEN_ADVANCE_BLIND" if self.post_started is not None else "GREEN_ADVANCE"
        return clamp(speed + steer, 0.0, MAX_SPEED), clamp(speed - steer, 0.0, MAX_SPEED), label

    def step(self, obs, now, can_start=True):
        if self.stopped:
            return self.stop(self.reason)
        positions, direction = obs.positions, obs.direction

        if self.state == "GREEN_STOP":
            # A second marker seen while stopped turns a left/right instruction into a dead end
            if direction == "BOTH":
                self.both_votes += 1
            if self.direction != "BOTH" and self.both_votes >= GREEN_BOTH_UPGRADE:
                self.direction = "BOTH"
            if now - self.state_started < GREEN_STOP_TIME:
                return 0.0, 0.0, "GREEN_STOP"
            if self.direction == "BOTH":
                self.begin_turn(now)    # dead end: turn around before entering the junction
            else:
                self.begin_advance(now)

        if self.state == "GREEN_ADVANCE":
            command = self.advance(obs, now)
            if command is not None:
                return command
            self.begin_turn(now)

        if self.state == "GREEN_TURN":
            elapsed = now - self.state_started
            limit = UTURN_TIMEOUT if self.direction == "BOTH" else GREEN_TURN_TIMEOUT
            minimum = UTURN_MIN_TIME if self.direction == "BOTH" else GREEN_TURN_MIN_TIME
            if elapsed >= limit:
                return self.stop("green turn timed out")
            # The line must first leave the centre, then come back and stay centred
            centered = line_centered(positions)
            if not centered:
                self.departed = True
            if self.departed and elapsed >= minimum and centered:
                self.reacquire_frames += 1
            else:
                self.reacquire_frames = 0
            if self.reacquire_frames < GREEN_REACQUIRE_FRAMES:
                return self.turn(positions, elapsed)
            self.enter("FOLLOW", now)
            self.cooldown_until = now + GREEN_COOLDOWN
            self.clear_frames = 0
            self.votes.clear()
            return 0.0, 0.0, "GREEN_DONE"

        # Not in a maneuver: re-arm after the cooldown, then collect votes
        if not self.armed:
            self.clear_frames = self.clear_frames + 1 if direction not in ("LEFT", "RIGHT", "BOTH") else 0
            if now >= self.cooldown_until and self.clear_frames >= GREEN_CLEAR_FRAMES:
                self.armed = True

        if self.armed and can_start:
            self.votes.append(direction)
            decided = self.confirmed()
            if decided is not None:
                self.direction = decided
                self.armed = False
                self.votes.clear()
                self.both_votes = 0
                self.enter("GREEN_STOP", now)
                return 0.0, 0.0, "GREEN_STOP"
        else:
            self.votes.clear()
        return None

## 7. NavigationCoordinator

The single place where the robot decides which controller is in charge. Each frame goes to the green-marker controller first. If it returns `None`, the frame goes to the gap controller, which passes ordinary frames on to the line controller.

A new green maneuver may not start during a gap crossing, since the robot has no line to steer by. It may start during a black-line pivot only if a junction is visible, because in that case the pivot was most likely caused by the crossbar. After every green turn the gap and line controllers are recreated, so that their history from before the turn is discarded.

Any stop is latched until the next run.

In [ ]:
class NavigationCoordinator:
    # Priority: active or confirmed green maneuver > gap crossing > black-line following

    def __init__(self):
        self.green_turn = GreenMarkerTurnController()
        self.gap = GapCrossingController()
        self.stopped = False
        self.reason = ""
        self.active_controller = "BLACK_LINE"   # shown in the status line and the log

    def stop(self, reason):
        self.stopped = True
        self.reason = reason
        self.active_controller = "STOPPED"
        return 0.0, 0.0, "STOP: " + reason

    def step(self, obs, now):
        if self.stopped:
            return self.stop(self.reason)

        line_turning = self.gap.black_line.turn_started is not None
        can_start_green = (self.gap.gap_started is None and
                           (not line_turning or obs.junction_y is not None))
        command = self.green_turn.step(obs, now, can_start=can_start_green)
        if command is not None:
            self.active_controller = "GREEN_TURN"
            if self.green_turn.stopped:
                self.stopped = True
                self.reason = self.green_turn.reason
                self.active_controller = "STOPPED"
            elif command[2] == "GREEN_DONE":
                self.gap = GapCrossingController()   # fresh line/gap controllers after a turn
            return command

        command = self.gap.step(obs.positions, now)
        self.active_controller = "GAP_CROSSING" if self.gap.gap_started is not None else "BLACK_LINE"
        if self.gap.black_line.stopped:
            return self.stop(command[2])
        return command

## 8. Main loop and logging

`run_v10()` is the main control loop. For each new camera frame it:

1. checks that the frame is less than 0.5 s old and has not already been processed,
2. runs perception and asks the coordinator for a motor command,
3. sends the command to the motors (only in drive mode),
4. stores one row of data in the log and, now and then, updates the on-screen preview.

The motors are stopped in a `finally` block, so they also stop if the code raises an error.

To keep the loop timing steady, nothing is written to disk while driving. Each frame adds one row to a fixed-size buffer in memory, and the CSV file is saved after the motors have stopped. The summary printed at the end gives the loop rate, the processing time per frame and a list of every state change, which shows which state ended the run.

In [ ]:
LOG_FIELDS = ("t", "frame_age", "proc_ms", "owner", "state", "marker", "near", "middle", "far",
              "junction_y", "zone_markers", "left", "right")
LOG = deque(maxlen=6000)   # about 3.5 minutes at 29 Hz; the oldest rows are dropped when full


def summarize_log(save=True):
    rows = list(LOG)
    if len(rows) < 2:
        print("No log data.")
        return

    # Timing statistics
    t = np.array([r[0] for r in rows]); proc = np.array([r[2] for r in rows]); age = np.array([r[1] for r in rows])
    periods = np.diff(t)
    print("frames %d | loop %.1f Hz (period mean %.0f ms, max %.0f ms) | processing mean %.1f ms, p95 %.1f ms, max %.1f ms | frame age max %.0f ms"
          % (len(rows), 1.0 / periods.mean(), 1000 * periods.mean(), 1000 * periods.max(),
             proc.mean(), np.percentile(proc, 95), proc.max(), 1000 * age.max()))

    # One line per change of controller or state
    previous = None
    for r in rows:
        key = (r[3], r[4].split(":")[0])
        if key != previous:
            print("  t=%6.2f  %-12s %-22s marker=%-7s junction=%s" % (r[0], r[3], r[4], r[5],
                  "-" if r[9] is None else "%.2f" % r[9]))
            previous = key

    if save:
        name = "run_%d.csv" % int(time.time())
        with open(name, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(LOG_FIELDS)
            writer.writerows(rows)
        print("Saved", name)

In [ ]:
overlay_view = widgets.Image(format="jpeg", width=336, height=336)
mask_view = widgets.Image(format="jpeg", width=336, height=336)
status_view = widgets.Label(value="Ready for stationary preview")
display(widgets.VBox([widgets.HBox([overlay_view, mask_view]), status_view]))


def run_v10(seconds=10, drive=False, show=True):
    robot.stop()
    controller = NavigationCoordinator()
    LOG.clear()
    start = time.monotonic()
    deadline = start + seconds
    previous_stamp = None
    next_display = 0.0
    # Updating the preview is not needed for control, so do it less often while driving
    display_period = 0.3 if drive else 0.1
    try:
        while time.monotonic() < deadline:
            sample = latest["sample"]
            if sample is None:
                print("Stopped: no camera frame")
                break
            frame, captured_at = sample
            now = time.monotonic()
            if now - captured_at > FRAME_TIMEOUT:
                print("Stopped: stale camera frame")
                break
            if captured_at == previous_stamp:   # same frame as last time
                time.sleep(0.005)
                continue
            previous_stamp = captured_at

            t0 = time.monotonic()
            obs, overlay, green_mask = perceive(frame)
            now = time.monotonic()
            proc_ms = 1000.0 * (now - t0)
            # Check freshness again: processing must not have made the frame too old
            if now >= deadline or now - captured_at > FRAME_TIMEOUT:
                print("Stopped: deadline or slow image processing")
                break

            left, right, state = controller.step(obs, now)
            if drive:
                robot.set_motors(left, right)

            p = obs.positions
            LOG.append((now - start, now - captured_at, proc_ms, controller.active_controller, state,
                        obs.direction, p.get("near"), p.get("middle"), p.get("far"),
                        obs.junction_y, obs.zone_markers, left, right))

            # Preview: line through the bands (magenta), markers (green), junction row (red)
            if show and now >= next_display:
                height, width = frame.shape[:2]
                points = get_line_points(p, width, height)
                for a, b in zip(points, points[1:]):
                    cv2.line(overlay, tuple(map(int, a)), tuple(map(int, b)), (255, 0, 255), 2)
                for marker in obs.markers:
                    x, y, w, h = marker["box"]
                    cv2.rectangle(overlay, (x, y), (x + w, y + h), (0, 255, 0), 2)
                if obs.junction_y is not None:
                    jy = int(obs.junction_y * height)
                    cv2.line(overlay, (0, jy), (width - 1, jy), (0, 0, 255), 1)
                overlay_view.value = bgr8_to_jpeg(overlay)
                mask_view.value = bgr8_to_jpeg(cv2.cvtColor(green_mask, cv2.COLOR_GRAY2BGR))
                status_view.value = "%s | %s | marker=%s | L=%.2f R=%.2f | owner=%s" % (
                    "DRIVE" if drive else "PREVIEW", state, obs.direction, left, right, controller.active_controller)
                next_display = now + display_period

            if controller.stopped:
                print(state)
                break
            time.sleep(0.005)
    finally:
        robot.stop()
        print("Run ended; motors stopped.")
        summarize_log()


def run_preview(seconds=20):
    # Same loop with the motors off; push the robot by hand to check perception
    run_v10(seconds=seconds, drive=False)


def run_robot(seconds=2):
    run_v10(seconds=seconds, drive=True)

## 9. Testing

Recommended order:

1. Run the perception self-test (Section 3.1).
2. Check the calibration values in Section 2 (rotation time and blind time).
3. Run `run_preview()` and push the robot by hand over a marked junction. The red line should appear on the crossbar and the status line should show the correct marker.
4. On the floor, test single features first (one left, right and dead-end marker, one bend, one gap), then longer sections of the course.

After each run, the printed list of state changes shows exactly which state ended the run.

**Stationary preview** (motors off):

In [ ]:
run_preview(seconds=40)

**Driving run.** The output below is from a 20 s floor run that included three green-marker turns.

In [ ]:
run_robot(seconds=40)

Run ended; motors stopped.
frames 589 | loop 29.4 Hz (period mean 34 ms, max 96 ms) | processing mean 3.7 ms, p95 4.8 ms, max 9.7 ms | frame age max 44 ms
  t=  0.01  BLACK_LINE   FOLLOW                 marker=NONE    junction=-
  t=  0.67  BLACK_LINE   FOLLOW_SLOW            marker=NONE    junction=0.55
  t=  0.88  BLACK_LINE   TURN                   marker=NONE    junction=0.81
  t=  1.34  BLACK_LINE   FOLLOW                 marker=NONE    junction=-
  t=  1.57  BLACK_LINE   FOLLOW_SLOW            marker=NONE    junction=-
  t=  1.77  BLACK_LINE   TURN                   marker=NONE    junction=-
  t=  2.10  BLACK_LINE   FOLLOW                 marker=NONE    junction=-
  t=  2.41  BLACK_LINE   FOLLOW_SLOW            marker=CENTER  junction=-
  t=  2.44  BLACK_LINE   TURN                   marker=CENTER  junction=0.52
  t=  2.64  GREEN_TURN   GREEN_STOP             marker=LEFT    junction=0.63
  t=  2.84  GREEN_TURN   GREEN_ADVANCE          marker=LEFT    junction=0.75
  t=  3.17  GREE

## 10. Cleanup

Stops the motors and releases the camera. Run this before opening another notebook that uses the camera.

In [ ]:
robot.stop()
camera.unobserve(receive_frame, names="value")
camera_link.unlink()
camera.stop()